# Using Alphafold components on Google Colab

---



In [1]:
#@title Install dependencies { vertical-output: true, display-mode: "form" }
#@markdown * Copied from the [Alphafold Colab notebook](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb)

from IPython.utils import io
import os
import subprocess
import tqdm.notebook

TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

GIT_REPO = 'https://github.com/deepmind/alphafold'
DATA_REPO = 'https://github.com/sameerd/data_dump'
TMSCORE_BIN = "./data_dump/TMscore/TMscore"

force_reinstall_dependencies = False  #@param {type:"boolean"}

if (not os.path.exists("INSTALLED_DEPS")) or force_reinstall_dependencies:
  try:
    with tqdm.notebook.tqdm(total=100, bar_format=TQDM_BAR_FORMAT) as pbar:
      with io.capture_output() as captured:
        %shell rm -f INSTALLED_DEPS
        # Uninstall default Colab version of TF.
        %shell pip uninstall -y tensorflow
        pbar.update(6)
        %shell rm -rf alphafold
        %shell git clone --branch main {GIT_REPO} alphafold
        pbar.update(8)
        %shell pip3 install -r ./alphafold/requirements.txt
        pbar.update(46)
        # Run setup.py to install only AlphaFold.
        %shell pip3 install --no-dependencies ./alphafold
        pbar.update(28)
        # add optax in case we want to use an optimizer
        %shell pip3 install --no-dependencies optax
        pbar.update(4)
        %shell rm -rf data_dump
        %shell git clone --branch main {DATA_REPO} data_dump
        pbar.update(4)
        %shell (cd data_dump/TMscore; make)
        pbar.update(4)
        %shell touch INSTALLED_DEPS
  except subprocess.CalledProcessError:
    print(captured)
    raise


import jax
if jax.local_devices()[0].platform == 'tpu':
  raise RuntimeError('Colab TPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
elif jax.local_devices()[0].platform == 'cpu':
  #raise RuntimeError('Colab CPU runtime not supported. Change it to GPU via Runtime -> Change Runtime Type -> Hardware accelerator -> GPU.')
  print(f"{jax.local_devices()}")
else:
  print(f'Running with {jax.local_devices()[0].device_kind} GPU')


# Make sure all necessary environment variables are set.
import os
os.environ['TF_FORCE_UNIFIED_MEMORY'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '2.0'





  0%|          | 0/100 [elapsed: 00:00 remaining: ?]

Running with Tesla T4 GPU


In [2]:
#@title  { display-mode: "form" }
from re import T
from typing import Any, Mapping, Optional, Union

import dataclasses
import gzip
import pathlib

import numpy as np
import jax.numpy as jnp

#@markdown ### Set global seed
SEED = 100 #@param 
q = 20

import alphafold
import alphafold.common
from alphafold.common.protein import from_pdb_string
from alphafold.common.residue_constants import restypes

def convert_aatype_to_string(prot_aatype : np.ndarray):
  # convert amino acid indices in protein to a sequence string
  return "".join(restypes[i] for i in prot_aatype)

def read_pdb_gz(filename):
  with gzip.open(filename, "rt") as fh:
    return from_pdb_string(fh.read())

#@markdown ### Read in the starting sequence
data_dir = pathlib.Path("data_dump")
starting_filename = "starting.pdb.gz" #@param
s_prot = read_pdb_gz(data_dir/ starting_filename)
starting_seq = convert_aatype_to_string(s_prot.aatype)
num_res = len(starting_seq)

print(f"Seq : {starting_seq}\nNum Residues : {num_res}")
for f in dataclasses.fields(s_prot):
  print(f"{f.name:15s}{f.type},  {getattr(s_prot, f.name).shape}")
print()

training_directory = "training_pdbs" #@param
training_pdb_list = list((data_dir / training_directory).glob("*.pdb.gz"))
N_all = len(training_pdb_list)
print(f"Num structures in dataset : {N_all}")

validation_fraction = 0.1 #@param
N_val = int(N_all * validation_fraction) 
N_train = N_all - N_val

np.random.seed(SEED)
train_indices = np.random.choice(N_all, size=N_train, replace=False)
all_data_mask = np.zeros(N_all, dtype=bool)
all_data_mask[train_indices] = True
val_indices = np.where(~all_data_mask)[0]

def get_data_for_train_index(train_index):
  filename = training_pdb_list[train_index]
  prot = read_pdb_gz(filename)
  return prot

assert(len(train_indices) == N_train)
assert(len(val_indices) == N_val)

print(f"Splitting dataset         : N_train: {N_train}, N_val={N_val}")


Seq : MQHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAIVMDIVPDGERDAWWDDEGFSSSPFTKNAHHAGIVATSVTLGQLQREQGDKLVSKAAEYFGIACRVNDGLRTTRFVRLFSDALDAKPLTIGHDYEVEFLLATRRVYEPFEAPFNFAPHCDDVSYGRDTVNWPLKRSFPRQLGGFLTIQGADNDAGMVMWDNRPESRAALDEMHAEYRETGAIAALERAAKIMLKPQPGQLTLFQSKNLHAIERCTSTRRTMGLFLIHTEDGWRMFD
Num Residues : 273
atom_positions <class 'numpy.ndarray'>,  (273, 37, 3)
aatype         <class 'numpy.ndarray'>,  (273,)
atom_mask      <class 'numpy.ndarray'>,  (273, 37)
residue_index  <class 'numpy.ndarray'>,  (273,)
chain_index    <class 'numpy.ndarray'>,  (273,)
b_factors      <class 'numpy.ndarray'>,  (273, 37)

Num structures in dataset : 1000
Splitting dataset         : N_train: 900, N_val=100


In [3]:
from alphafold.common import residue_constants
from alphafold.model import quat_affine
from alphafold.model import all_atom
from alphafold.model import r3

from alphafold.model import config
model_name = ('model_1')
cfg = config.model_config(model_name)
cfg.data.eval.num_ensemble = 1

## Testing pipeline

In [4]:
v_prot = read_pdb_gz(training_pdb_list[train_indices[0]])
v_seq = convert_aatype_to_string(v_prot.aatype)

In [5]:
v_frames = all_atom.atom37_to_frames(v_prot.aatype, v_prot.atom_positions, v_prot.atom_mask)
s_frames = all_atom.atom37_to_frames(s_prot.aatype, s_prot.atom_positions, s_prot.atom_mask)

In [6]:
v_bb_frames = v_frames["rigidgroups_gt_frames"][:, 0, :]
s_bb_frames = s_frames["rigidgroups_gt_frames"][:, 0, :]

In [7]:
all_atom.frame_aligned_point_error(
    pred_frames = r3.rigids_from_tensor_flat12(s_bb_frames),
    target_frames = r3.rigids_from_tensor_flat12(v_bb_frames),
    frames_mask = v_frames["rigidgroups_gt_exists"][:, 0], # should be all ones
    pred_positions = r3.rigids_from_tensor_flat12(s_bb_frames).trans,
    target_positions = r3.rigids_from_tensor_flat12(v_bb_frames).trans,
    positions_mask = v_frames["rigidgroups_gt_exists"][:, 0], # should be all ones
    l1_clamp_distance = cfg.model.heads.structure_module.fape.clamp_distance,
    length_scale = cfg.model.heads.structure_module.fape.loss_unit_distance
)

DeviceArray(0.3037586, dtype=float32)

In [8]:
all_atom.frame_aligned_point_error(
    pred_frames = r3.rigids_from_tensor_flat12(v_bb_frames),
    target_frames = r3.rigids_from_tensor_flat12(v_bb_frames),
    frames_mask = v_frames["rigidgroups_gt_exists"][:, 0], # should be all ones
    pred_positions = r3.rigids_from_tensor_flat12(v_bb_frames).trans,
    target_positions = r3.rigids_from_tensor_flat12(v_bb_frames).trans,
    positions_mask = v_frames["rigidgroups_gt_exists"][:, 0], # should be all ones
    l1_clamp_distance = cfg.model.heads.structure_module.fape.clamp_distance,
    length_scale = cfg.model.heads.structure_module.fape.loss_unit_distance
)

DeviceArray(0.001, dtype=float32)

In [9]:
from alphafold.model.modules import dgram_from_positions
from alphafold.model.tf.data_transforms import pseudo_beta_fn

dgram_features = cfg.model.embeddings_and_evoformer.template.dgram_features
print(dgram_features)

# create pseudo beta for glycines # convert tensorflow array to numpy
pseudo_beta_pos = pseudo_beta_fn(s_prot.aatype, s_prot.atom_positions, 
                                 all_atom_masks = None).numpy() 
s_dgram = dgram_from_positions(pseudo_beta_pos, **dgram_features)
s_dgram.shape

{max_bin: 50.75, min_bin: 3.25, num_bins: 39}



(273, 273, 39)

In [10]:
n, ca, c = [residue_constants.atom_order[a] for a in ('N', 'CA', 'C')]
rot, trans = quat_affine.make_transform_from_reference(
        n_xyz=s_prot.atom_positions[:, n],
        ca_xyz=s_prot.atom_positions[:, ca],
        c_xyz=s_prot.atom_positions[:, c])
affines = quat_affine.QuatAffine(
        quaternion=quat_affine.rot_to_quat(rot, unstack_inputs=True),
        translation=trans,
        rotation=rot,
        unstack_inputs=True)
points = [jnp.expand_dims(x, axis=-2) for x in affines.translation]
affine_vec = affines.invert_point(points, extra_dims=1)
backb_to_global = r3.rigids_from_quataffine(affines)


In [11]:
# make features
import alphafold.data.pipeline as pipeline
import alphafold.model.features as features

features_dict = pipeline.make_sequence_features(starting_seq, description="query", num_res=num_res)
del features_dict['between_segment_residues']
del features_dict['domain_name']
del features_dict['sequence']
features_dict

{'aatype': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=int32),
 'residue_index': array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131,

## Model

In [12]:
cfg.model.heads.structure_module.fape

clamp_distance: 10.0
clamp_type: relu
loss_unit_distance: 10.0

In [13]:
import functools
from alphafold.model import common_modules
import haiku as hk

def get_bb_frames_from_prot(p):
    gt_frames = all_atom.atom37_to_frames(p.aatype, p.atom_positions, 
                                          p.atom_mask)
    gt_bb_frames = gt_frames["rigidgroups_gt_frames"][:, 0, :]
    return gt_bb_frames

class PredictBB(hk.Module):
  """ Predict the backbone Quaternion and translation
      Jumper et al. (2021) Suppl. Algorithm 23.
   """

  def __init__(self,
               config,
               num_channels,
               name="predict_bb"):
    super().__init__(name=name)
    self.config = config
    self.num_channels = num_channels

  def __call__(self, batch, is_training=True):
    """
      batch : dict with training data 
    """
    aatype = batch['aatype'] # one-hot encoded sequence
    num_res = aatype.shape[0]
    fape_cfg = cfg.model.heads.structure_module.fape

    # first embedd the 20 amino acids into num_channels
    seq_embedder = common_modules.Linear(num_output=self.num_channels, 
                         num_input_dims=1, name="one_hot_projection")
    seq_embedding = seq_embedder(aatype)
    relu = jax.nn.relu 
    # relu activations
    act = relu( seq_embedding)
    #act = jax.nn.softmax(act)

    # affine prediction
    affine_pred_size = 7 # 4 quarternions + 3 for translation
    affine_preds = common_modules.Linear(affine_pred_size,
                                         name="affine_updates")(act)
    # convert affine to rigids
    qa = quat_affine.QuatAffine.from_tensor(affine_preds)
    rigids = r3.rigids_from_quataffine(qa)

    gt_bb_frames = batch["gt_bb_frames"]
    gt_rigids = r3.rigids_from_tensor_flat12(gt_bb_frames)

    fape_loss_fn = functools.partial(
      all_atom.frame_aligned_point_error,
      l1_clamp_distance=fape_cfg.clamp_distance,
      length_scale=fape_cfg.loss_unit_distance)
    
    fape = fape_loss_fn(
      pred_frames = rigids,
      target_frames = gt_rigids,
      frames_mask = jnp.ones(num_res), # should be all ones
      pred_positions = rigids.trans,
      target_positions = gt_rigids.trans,
      positions_mask = jnp.ones(num_res), # should be all ones
    )

    return fape

def _forward_fn_predictBB(x):
  module = PredictBB(config=cfg, num_channels=50)
  return module(x)

forward_predictBB = hk.transform(_forward_fn_predictBB)
forward_predictBB = hk.without_apply_rng(forward_predictBB)

jit_forward_predictBB_apply = jax.jit(forward_predictBB.apply)
jit_grad_forward_predictBB_apply = jax.jit(jax.grad(jit_forward_predictBB_apply))

In [14]:
batch = {}

batch['aatype'] = jax.nn.one_hot(s_prot.aatype, 20)
batch["gt_bb_frames"] = get_bb_frames_from_prot(s_prot)

bigbatch = {}
bigbatch["aatype"] = jnp.stack([batch["aatype"], batch["aatype"]])
bigbatch["gt_bb_frames"] = jnp.stack([batch["gt_bb_frames"], batch["gt_bb_frames"]])

batch_data = batch
rng_key = jax.random.PRNGKey(SEED)
params = forward_predictBB.init(x=batch_data, rng=rng_key)
print("one_hot_projects.weights.shape :", 
      params['predict_bb/one_hot_projection']['weights'].shape)

output = jit_forward_predictBB_apply(params=params, x=batch_data)
print("output.shape :", output.shape)
fape = output
print(fape)

one_hot_projects.weights.shape : (20, 50)
output.shape : ()
0.9787863


In [15]:
from jax.config import config
config.update("jax_debug_nans", True)

import operator

In [16]:
batch_size = 90
num_epochs = 2
lr = 0.01 


@jax.jit
def update_rule(param, grads):
  return param - lr * grads

# clip our gradients
jnp_clip = functools.partial(jnp.nan_to_num, nan=0, posinf=10, neginf=-10)


for epoch in range(num_epochs):
  epoch_train_indices = train_indices.copy()
  np.random.shuffle(epoch_train_indices)
  for batch_num in tqdm.notebook.tqdm(range(len(epoch_train_indices) // batch_size)):
    batch_indices = epoch_train_indices[
            (batch_num*batch_size):((batch_num+1)*batch_size)]
    batch_grads = None
    batch_losses = []
    for batch_idx in batch_indices:
      try:
        prot = get_data_for_train_index(batch_idx)
      except KeyboardInterrupt:
        raise
      except:
        print("Cannot load protein idx : ", batch_idx)
        continue
      batch = {}
      batch["aatype"] = jax.nn.one_hot(prot.aatype, num_classes=q)
      batch["gt_bb_frames"] = get_bb_frames_from_prot(prot)
      this_grad = jit_grad_forward_predictBB_apply(params, x=batch)
      this_grad = jax.tree_map(jnp_clip, this_grad)
      this_loss = jit_forward_predictBB_apply(params, x=batch)
      
      if jnp.stack(jax.tree_leaves( # check if we have any bad gradients
          jax.tree_map(lambda x: jnp.isnan(x).any(), this_loss))).any():
        raise
      if batch_grads is None:
        batch_grads = this_grad
      else:
        batch_grads = jax.tree_multimap(operator.add, batch_grads, this_grad)
      batch_losses.append(this_loss)


    avg_losses = jnp.mean(jnp.stack(batch_losses))
    print(batch_num, avg_losses)
    params = jax.tree_multimap(update_rule, params, batch_grads)


  0%|          | 0/10 [00:00<?, ?it/s]

0 0.97908074
1 0.97908235
2 0.9790743
3 0.97907794
4 0.97907734
5 0.9790816
6 0.9790801
7 0.9790803
8 0.97907853
9 0.9790727


  0%|          | 0/10 [00:00<?, ?it/s]

0 0.9790845
1 0.9790833
2 0.9790775
3 0.97908276
4 0.97907686
5 0.9790737
6 0.97907495
7 0.97908056
8 0.97907954
9 0.9790719


In [17]:
import logging
import alphafold.model.folding
import tensorflow.compat.v1 as tf

In [18]:
%shell {TMSCORE_BIN}


 Brief instruction for running TM-score program:
 (For detail: Zhang & Skolnick, Proteins, 2004 57:702-10)

 1. Run TM-score to compare 'model.pdb' and 'native.pdb':
     $ TMscore model.pdb native.pdb

 2. Run TM-score to compare two complex structures with multiple chains
     $ TMscore -c model.pdb native.pdb

 3. TM-score normalized with an assigned scale d0 e.g. 5 A:
     $ TMscore model.pdb native.pdb -d 5

 4. TM-score normalized by a specific length, e.g. 120 AA:
     $ TMscore model.pdb native.pdb -l 120

 5. TM-score with superposition output, e.g. 'TM_sup*':
     $ TMscore model.pdb native.pdb -o TM_sup
    View superposed CA-traces by RasMol or PyMOL:
     $ rasmol -script TM_sup
     $ pymol -d @TM_sup.pml
    View superposed atomic models by RasMol:
     $ rasmol -script TM_sup_atm
     $ pymol -d @TM_sup_atm.pml

 6. For full help message:
    $ TMscore -h

